For multiclass from scratch,multiclass ,Mnist dataset,use 5 layers,hyperparameter, epochs atleast 50

In [2]:
!pip install numba opencv-python matplotlib

In [3]:
import numpy as np
import cv2
import os
import math
from numba import cuda
import matplotlib.pyplot as plt

print("CUDA Available:", cuda.is_available())

CUDA Available: True


In [4]:
EPOCHS = 50
LR = 0.01
BATCH_SIZE = 64

CONV1_FILTERS = 8
CONV2_FILTERS = 16
KERNEL_SIZE = 3
HIDDEN1 = 128
HIDDEN2 = 64
NUM_CLASSES = 10

In [5]:
import numpy as np
from numba import cuda

@cuda.jit
def conv2d(input_img, kernel, output):
    i, j, k = cuda.grid(3)

    if (i < output.shape[0] and
        j < output.shape[1] and
        k < output.shape[2]):

        val = 0.0
        for ki in range(kernel.shape[1]):
            for kj in range(kernel.shape[2]):
                val += input_img[i+ki, j+kj] * kernel[k, ki, kj]

        output[i, j, k] = val

In [6]:
def relu(x):
    return np.maximum(0, x)

def softmax(x):
    exp = np.exp(x - np.max(x))
    return exp / np.sum(exp)

def cross_entropy(pred, label):
    return -np.log(pred[label] + 1e-9)

In [7]:
class Dense:
    def __init__(self, inp, out):
        self.W = np.random.randn(inp, out) * 0.01
        self.b = np.zeros(out)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, grad, lr):
        dW = np.outer(self.x, grad)
        db = grad
        dx = self.W @ grad

        self.W -= lr * dW
        self.b -= lr * db
        return dx

In [8]:
class ConvLayer:
    def __init__(self, in_channels, out_channels, k):
        self.kernels = np.random.randn(out_channels, k, k) * 0.01

    def forward(self, x):
        h, w = x.shape
        k = self.kernels.shape[1]
        out_h = h - k + 1
        out_w = w - k + 1

        output = np.zeros((out_h, out_w, self.kernels.shape[0]))

        d_x = cuda.to_device(x.astype(np.float32))
        d_k = cuda.to_device(self.kernels.astype(np.float32))
        d_out = cuda.to_device(output.astype(np.float32))

        threads = (8, 8, 1)
        blocks = (
            (out_h+7)//8,
            (out_w+7)//8,
            self.kernels.shape[0]
        )

        conv2d[blocks, threads](d_x, d_k, d_out)
        return d_out.copy_to_host()

In [9]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np

mnist = fetch_openml('mnist_784', version=1)

# convert DataFrame → numpy
X = mnist.data.to_numpy().reshape(-1, 28, 28).astype(np.float32) / 255
y = mnist.target.astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

In [10]:
conv1 = ConvLayer(1, CONV1_FILTERS, KERNEL_SIZE)
conv2 = ConvLayer(CONV1_FILTERS, CONV2_FILTERS, KERNEL_SIZE)

fc1 = Dense(24*24*CONV2_FILTERS, HIDDEN1)
fc2 = Dense(HIDDEN1, HIDDEN2)
fc3 = Dense(HIDDEN2, NUM_CLASSES)

In [11]:
for epoch in range(EPOCHS):
    loss_total = 0

    for img, label in zip(X_train, y_train):

        # ----- Forward -----
        x = conv1.forward(img)
        x = relu(x)

        x = conv2.forward(x[:,:,0])
        x = relu(x)

        x = x.reshape(-1)

        x = relu(fc1.forward(x))
        x = relu(fc2.forward(x))
        logits = fc3.forward(x)

        probs = softmax(logits)
        loss = cross_entropy(probs, label)
        loss_total += loss

        # ----- Backprop -----
        grad = probs
        grad[label] -= 1

        grad = fc3.backward(grad, LR)
        grad = fc2.backward(grad, LR)
        grad = fc1.backward(grad, LR)

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss {loss_total/len(X_train):.4f}")

Epoch 1/50, Loss 2.3033
Epoch 2/50, Loss 2.3033
Epoch 3/50, Loss 2.3033
Epoch 4/50, Loss 2.3033
Epoch 5/50, Loss 2.3033


KeyboardInterrupt: 

In [1]:
def predict(img):
    x = conv1.forward(img)
    x = relu(x)

    x = conv2.forward(x[:,:,0])
    x = relu(x)

    x = x.reshape(-1)
    x = relu(fc1.forward(x))
    x = relu(fc2.forward(x))
    x = fc3.forward(x)

    return np.argmax(softmax(x))